# <center>YOLO 前处理详解</center>

在目标检测任务中，**YOLO（You Only Look Once）** 系列算法以其高效性和准确性而备受青睐。然而，YOLO 模型的性能不仅取决于其网络结构的设计，还与其前处理过程密切相关。前处理阶段的核心任务是将输入图像调整为模型所需的格式，同时最大限度地保留图像的关键信息。本节课程将深入探讨 YOLO 前处理中的关键技术，涵盖 **Letterbox**、**仿射变换与双线性插值**，以及**基于 CUDA 的 WarpAffine 和 MultiWarpAffine 实现**。

YOLO 的前处理通常包括以下步骤：

- **等比例缩放**：保持图像纵横比的同时将图像缩放到目标尺寸。
- **边框填充**：在图像周围添加填充，以确保缩放后的图像符合目标尺寸。
- **转换为浮点数**：将图像像素值从整数转换为浮点数，便于后续计算。
- **归一化**：对图像像素值进行归一化处理，通常将其缩放到 [0, 1] 或 [-1, 1] 范围内。
- **通道转换**：将图像数据格式从 HWC（高度、宽度、通道）转换为 CHW（通道、高度、宽度），以适应模型输入要求。

通过以上步骤，YOLO 的前处理能够为模型提供高质量的输入数据，从而提升检测性能。

## <center>Letterbox 源码解析</center>

In [ ]:
import cv2
import numpy as np
from typing import Tuple

def letterbox(img: np.ndarray, new_shape: Tuple = (640, 640)) -> Tuple[np.ndarray, Tuple[float, float]]:
    """缩放图片同时保持纵横比并添加填充，适用于 YOLO 模型。"""
    shape = img.shape[:2]  # 当前图片的高度和宽度 [height, width]

    # 新旧尺寸之间的缩放比例（新尺寸 / 旧尺寸）
    r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])

    # 计算填充量
    new_unpad = int(round(shape[1] * r)), int(round(shape[0] * r))  # 缩放后未填充的尺寸
    dw, dh = (new_shape[1] - new_unpad[0]) / 2, (new_shape[0] - new_unpad[1]) / 2  # 宽和高的填充量

    if shape[::-1] != new_unpad:  # 如果需要调整大小
        img = cv2.resize(img, new_unpad, interpolation=cv2.INTER_LINEAR)  # 使用线性插值调整图片大小到缩放后的尺寸

    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))  # 顶部和底部填充量
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))  # 左右填充量
    img = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(114, 114, 114))  # 用固定值填充图片边界

    # 返回图片和填充宽度、高度的比例（相对图片尺寸）
    return img, (top / img.shape[0], left / img.shape[1])

In [ ]:
from course_functions.ploting import compare_images

img_ori = cv2.imread('./course_data/images/img_example.jpg')
img_letterbox, _ = letterbox(img_ori, (640, 640))
compare_images(img_ori, img_letterbox, titles=["Original Image", "Letterbox Image"])

## <center>仿射变换</center>

`letterbox` 的核心功能是在保持图片纵横比的同时，将图片缩放到目标尺寸，并在多余的空间中添加填充。这一功能可以通过仿射变换实现，因为仿射变换能够同时处理缩放和平移操作。我们只需计算出仿射变换矩阵 $M$，即可通过 `cv2.warpAffine` 实现与 `letterbox` 相同的效果。以下是仿射变换矩阵 $M$ 的数学推理流程：

首先，计算缩放比例 $r$，以确保图像在缩放后不会超过目标尺寸的任一维度，同时保持纵横比。

$$
r = \min\left(\frac{\text{dst_width}}{\text{src_width}}, \frac{\text{dst_height}}{\text{src_height}}\right)
$$

接下来，计算在宽度和高度方向上的填充量 $dw$ 和 $dh$ 。这些填充量将用于在图像的两侧和上下添加填充，以使图像居中。

$$
\text{new_width} = \text{src_width} \times r, \text{new_height} = \text{src_height} \times r
$$

$$
dw = \frac{\text{dst_width} - \text{new_width}}{2}, dh = \frac{\text{dst_height} - \text{new_height}}{2}
$$

线性变换矩阵用于缩放图像。由于我们需要在 $x$ 和 $y$ 方向上以相同的比例 $r$ 缩放图像，线性变换矩阵 $A$ 为：

$$
M = \begin{pmatrix}
r & 0 \\
0 & r
\end{pmatrix}
$$

平移向量用于在 $x$ 和 $y$ 方向上分别平移 $dw$ 和 $dh$，以使图像居中。平移向量 $t$ 为：

$$
M = \begin{pmatrix}
dw \\
dh
\end{pmatrix}
$$

仿射变换矩阵 $M$ 由线性变换矩阵 $A$ 和平移向量 $t$ 组成。将 $A$ 和 $t$ 合并，得到 $M$：

$$
M=\begin{pmatrix} A & t \end{pmatrix}=\begin{pmatrix}
r & 0 & dw \\
0 & r & dh
\end{pmatrix}
$$

逆矩阵 $M^{-1}$ 用于将目标图像（$dst$）的坐标映射回源图像（$src$）的坐标:

$$
M^{-1}=\begin{pmatrix}
\frac{1}{r} & 0 & -\frac{dw}{r} \\
0 & \frac{1}{r} & -\frac{dh}{r}
\end{pmatrix}
$$


In [ ]:
def warp_affine(img: np.ndarray, new_shape: Tuple = (320, 320)) -> Tuple[np.ndarray, Tuple[float, float]]:
    """
    使用 cv2.warpAffine 实现 letterbox 功能：缩放图片同时保持纵横比并添加填充，适用于 YOLO 模型。
    """
    shape = img.shape[:2]  # 当前图片的高度和宽度 [height, width]

    # 新旧尺寸之间的缩放比例（新尺寸 / 旧尺寸）
    r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])

    # 计算填充量
    new_unpad = int(round(shape[1] * r)), int(round(shape[0] * r))  # 缩放后未填充的尺寸
    dw, dh = (new_shape[1] - new_unpad[0]) / 2, (new_shape[0] - new_unpad[1]) / 2  # 宽和高的填充量

    # 构造仿射变换矩阵
    M = np.float32([[r, 0, dw], [0, r, dh]])

    # 应用仿射变换
    img_warped = cv2.warpAffine(img, M, dsize=(new_shape[1], new_shape[0]), borderValue=(114, 114, 114))

    # 返回图片和填充宽度、高度的比例（相对图片尺寸）
    return img_warped, (dh / new_shape[0], dw / new_shape[1])

In [ ]:
img_warp_affine, _ = warp_affine(img_ori, (640, 640)) # 调用 letterbox 方法缩放和填充图片
compare_images(img_ori, img_warp_affine, titles=["Original Image", "WarpAffine Image"])

### <center>仿射变换 CUDA 实现</center>

在通过 CUDA 实现仿射变换之前，我们需要深入理解如何将原始坐标通过仿射变换矩阵转换为变换后的坐标。仿射变换可以表示为：

$$
\begin{pmatrix}
x' \\
y'
\end{pmatrix}
=
\begin{pmatrix}
a & b \\
c & d
\end{pmatrix}
\begin{pmatrix}
x \\
y
\end{pmatrix}
+
\begin{pmatrix}
e \\
f
\end{pmatrix}
$$

其中，$\begin{pmatrix} x' \\ y' \end{pmatrix}$ 是变换后的坐标，$\begin{pmatrix} x \\ y \end{pmatrix}$ 是原始坐标，$\begin{pmatrix} a & b \\ c & d \end{pmatrix}$ 是线性变换矩阵，用于旋转、缩放、剪切等操作，$\begin{pmatrix} e \\ f \end{pmatrix}$ 是平移向量，用于平移操作。

为了将线性变换和平移统一表示，可以使用齐次坐标。齐次坐标将二维坐标 $(x, y)$ 扩展为三维坐标 $(x, y, 1)$，并将仿射变换表示为矩阵乘法：

$$
\begin{pmatrix}
x' \\
y' \\
1
\end{pmatrix}
=
\begin{pmatrix}
a & b & e \\
c & d & f \\
0 & 0 & 1
\end{pmatrix}
\begin{pmatrix}
x \\
y \\
1
\end{pmatrix}
$$

通过这种方式，我们可以将仿射变换统一表示为矩阵乘法，从而显著简化计算过程。接下来，我们将[利用 CUDA 实现仿射变换](./course_codes/02.01.01.WarpAffine)，充分发挥 GPU 的并行计算优势，大幅提升处理效率。 在此之前，让我们先了解一下基于 `Numpy` 的仿射变换实现方式。

In [ ]:
def warp_affine_manual(img: np.ndarray, new_shape: Tuple = (320, 320)) -> Tuple[np.ndarray, Tuple[float, float]]:
    """
    手动通过 WarpAffine 实现 letterbox 功能：缩放图片同时保持纵横比并添加填充，适用于 YOLO 模型。
    """
    shape = img.shape[:2]  # 当前图片的高度和宽度 [height, width]

    # 新旧尺寸之间的缩放比例（新尺寸 / 旧尺寸）
    r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])
    # inv_r = max(shape[0] / new_shape[0], shape[1] / new_shape[1])

    new_unpad = int(round(shape[1] * r)), int(round(shape[0] * r))  # 缩放后未填充的尺寸
    dw, dh = (new_shape[1] - new_unpad[0]) / 2, (new_shape[0] - new_unpad[1]) / 2  # 宽和高的填充量

    # 构造仿射变换矩阵
    M = np.float32([[r, 0, dw], [0, r, dh], [0, 0, 1]])

    # 求逆矩阵，也就是 new_shape -> shape 的矩阵
    INV_M = np.linalg.inv(M)
    # 等效为 以下结果
    # INV_M = np.float32([[1/r, 0, - (dw/r)], [0, 1/r, -(dh/r)], [0, 0, 1]])
    # INV_M = np.float32([[inv_r, 0, - dw * inv_r], [0, inv_r, - dh * inv_r], [0, 0, 1]])

    # 创建目标图像
    img_warped = np.full((new_shape[0], new_shape[1], 3), (114, 114, 114), dtype=np.uint8)

    # 遍历目标图像的每个像素
    for dy in range(new_shape[0]):
        for dx in range(new_shape[1]):
            # 使用逆变换矩阵计算原图像中的坐标
            x = INV_M[0, 0] * dx + INV_M[0, 1] * dy + INV_M[0, 2]
            y = INV_M[1, 0] * dx + INV_M[1, 1] * dy + INV_M[1, 2]

            # 检查是否在原图像范围内
            if 0 <= x < shape[1] and 0 <= y < shape[0]:
                # 最近邻插值
                u, v = int(round(x)), int(round(y))
                # 该代码实现的简易WarpAffine仅适用于用大尺寸->小尺寸，不然会出现 IndexError
                # 在实际的应用场景中几乎不会存在将输入画面上采样后作为模型的输入
                img_warped[dy, dx] = img[v, u] 

    # 返回图片和填充宽度、高度的比例（相对图片尺寸）
    return img_warped, (dh / new_shape[0], dw / new_shape[1])

In [ ]:
img_warp_affine_manual, _ = warp_affine_manual(img_ori, (640, 640)) # 调用 letterbox 方法缩放和填充图片
compare_images(img_ori, img_warp_affine_manual, titles=["Original Image", "WarpAffine Image"])

### <center>使用双线性插值提高精度</center>

在图像处理中，最近邻插值虽然计算简单，但会导致图像信息丢失，从而降低模型推理的精度。为了解决这个问题，我们采用双线性插值方法来保持图像的细节信息，提升图像质量，进而提高模型的检测性能。双线性插值的具体步骤如下：

假设我们想要得到未知函数 $f$ 在点 $P = (x, y)$ 的值，并且已知函数 $f$ 在四个点 $Q_{11} = (x_1, y_1)$、$Q_{12} = (x_1, y_2)$、$Q_{21} = (x_2, y_1)$ 和 $Q_{22} = (x_2, y_2)$ 的值。

首先，在 $x$ 方向进行线性插值，得到：

$$
\begin{aligned}
f(x, y_1) &\approx \frac{x_2 - x}{x_2 - x_1} f(Q_{11}) + \frac{x - x_1}{x_2 - x_1} f(Q_{21}), \\
f(x, y_2) &\approx \frac{x_2 - x}{x_2 - x_1} f(Q_{12}) + \frac{x - x_1}{x_2 - x_1} f(Q_{22}).
\end{aligned}
$$

接着，在 $y$ 方向进行线性插值，得到：

$$
\begin{aligned}
f(x, y) &\approx \frac{y_2 - y}{y_2 - y_1} f(x, y_1) + \frac{y - y_1}{y_2 - y_1} f(x, y_2) \\
&= \frac{y_2 - y}{y_2 - y_1} \left( \frac{x_2 - x}{x_2 - x_1} f(Q_{11}) + \frac{x - x_1}{x_2 - x_1} f(Q_{21}) \right) \\
&\quad + \frac{y - y_1}{y_2 - y_1} \left( \frac{x_2 - x}{x_2 - x_1} f(Q_{12}) + \frac{x - x_1}{x_2 - x_1} f(Q_{22}) \right) \\
&= \frac{1}{(x_2 - x_1)(y_2 - y_1)} \Big( f(Q_{11})(x_2 - x)(y_2 - y) + f(Q_{21})(x - x_1)(y_2 - y) \\
&\quad + f(Q_{12})(x_2 - x)(y - y_1) + f(Q_{22})(x - x_1)(y - y_1) \Big) \\
&= \frac{1}{(x_2 - x_1)(y_2 - y_1)} \begin{bmatrix} x_2 - x & x - x_1 \end{bmatrix} \begin{bmatrix} f(Q_{11}) & f(Q_{12}) \\ f(Q_{21}) & f(Q_{22}) \end{bmatrix} \begin{bmatrix} y_2 - y \\ y - y_1 \end{bmatrix}.
\end{aligned}
$$

需要注意的是，如果先在 $y$ 方向进行插值，再在 $x$ 方向进行插值，最终的结果与上述顺序的双线性插值结果是一致的。接下来，我们将[利用 CUDA 实现使用双线性插值的仿射变换](./course_codes/02.01.02.BilinearWarpAffine)，充分发挥 GPU 的并行计算优势，提升图像处理精度和效率。

### <center>多Batch仿射变换核函数</center>

在之前的仿射变换实现中，每次只能对单个图像进行处理。然而，在实际应用中，模型的输入通常是多个图像（即 batch 大于 1）。此时若逐个启动核函数进行仿射变换，会导致 CUDA 核函数启动多次，从而增加不必要的耗时。为了提高效率，我们实现了一个[支持多Batch的仿射变换核函数](./course_codes/02.01.03.MutliBilinearWarpAffine)，使得能够同时处理多个图像（假设输入尺寸固定），显著减少了核函数启动的开销，大幅提升处理效率，为大规模图像处理提供了有力支持。

## <center>总结</center>

本节课程深入剖析了 YOLO 前处理的各个环节，从 **Letterbox** 的实现原理到 **仿射变换** 的数学推理，再到 **双线性插值** 的精度提升策略，以及基于 **CUDA** 的高效并行实现。通过这些关键技术的讲解，我们展示了如何将输入图像精准地调整为模型所需的格式，同时最大限度地保留图像的关键信息，为模型的高效训练和准确预测奠定坚实基础。通过本节的学习，希望大家能够深刻理解 YOLO 前处理的重要性和实现细节，掌握如何通过科学的前处理方法提升目标检测模型的性能，为后续的模型开发和优化提供有力支持。